# Exercise 3: Memory Systems for Agents

In this exercise, you will learn to:
1. Implement different types of conversation memory
2. Use LangGraph checkpointing for persistent memory
3. Build a personalized assistant that remembers users
4. Combine short-term and long-term memory strategies

**Prerequisites**: Complete Exercises 1 and 2 first.

## Setup

In [1]:
# Install required packages
!pip install langchain langchain-openai langgraph --quiet


[notice] A new release of pip available: 22.3.1 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [7]:
import os
from dotenv import load_dotenv

load_dotenv()

# Verify that the required environment variables are set
assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY is not set! Check your .env file."
assert os.environ.get("OPENAI_ENDPOINT"), "OPENAI_ENDPOINT is not set! Check your .env file."

print("Environment variables loaded successfully!")

Environment variables loaded successfully!


In [12]:
from typing import Annotated, TypedDict, Literal
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, BaseMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import MemorySaver
from langgraph.prebuilt import ToolNode
import json

# Initialize the LLM
llm = ChatOpenAI(base_url=os.environ["OPENAI_ENDPOINT"], model="model-group3")

## Part 1: Understanding Memory in Agents

Memory allows agents to:
- Remember previous conversations
- Learn user preferences
- Maintain context across interactions
- Build on previous work

In [9]:
# Let's visualize the memory types
print("""
┌─────────────────────────────────────────────────────────────┐
│                    MEMORY TYPES                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────────┐                                        │
│  │ BUFFER MEMORY   │  Store all messages                    │
│  │ (Short-term)    │  ✓ Simple  ✗ Can exceed context        │
│  └─────────────────┘                                        │
│                                                             │
│  ┌─────────────────┐                                        │
│  │ WINDOW MEMORY   │  Store last N messages                 │
│  │ (Short-term)    │  ✓ Bounded  ✗ Loses old context        │
│  └─────────────────┘                                        │
│                                                             │
│  ┌─────────────────┐                                        │
│  │ SUMMARY MEMORY  │  Compress to summaries                 │
│  │ (Medium-term)   │  ✓ Compact  ✗ May lose details         │
│  └─────────────────┘                                        │
│                                                             │
│  ┌─────────────────┐                                        │
│  │ ENTITY MEMORY   │  Track specific entities               │
│  │ (Long-term)     │  ✓ Personalized  ✗ Complex             │
│  └─────────────────┘                                        │
│                                                             │
└─────────────────────────────────────────────────────────────┘
""")


┌─────────────────────────────────────────────────────────────┐
│                    MEMORY TYPES                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  ┌─────────────────┐                                        │
│  │ BUFFER MEMORY   │  Store all messages                    │
│  │ (Short-term)    │  ✓ Simple  ✗ Can exceed context        │
│  └─────────────────┘                                        │
│                                                             │
│  ┌─────────────────┐                                        │
│  │ WINDOW MEMORY   │  Store last N messages                 │
│  │ (Short-term)    │  ✓ Bounded  ✗ Loses old context        │
│  └─────────────────┘                                        │
│                                                             │
│  ┌─────────────────┐                                        │
│  │ SUMMARY MEMORY  │  Compress to sum

## Part 2: Basic Conversation Memory (Buffer)

Let's start with the simplest form - keeping all messages in memory.

In [13]:
# Simple state with message buffer
class BufferMemoryState(TypedDict):
    messages: Annotated[list, add_messages]

# Agent node
def buffer_agent(state: BufferMemoryState) -> dict:
    """Agent with full conversation history."""
    messages = state["messages"]
    
    # Add system message
    full_messages = [
        SystemMessage(content="You are a helpful assistant. You have access to the full conversation history."),
        *messages
    ]
    
    response = llm.invoke(full_messages)
    return {"messages": [response]}

# Build the graph
buffer_graph = StateGraph(BufferMemoryState)
buffer_graph.add_node("agent", buffer_agent)
buffer_graph.add_edge(START, "agent")
buffer_graph.add_edge("agent", END)

# Compile WITHOUT checkpointing first
buffer_app_no_memory = buffer_graph.compile()

print("Buffer memory graph compiled (no persistence)!")

Buffer memory graph compiled (no persistence)!


In [14]:
# Test without persistence - memory is lost between calls
print("Test WITHOUT persistence:")
print("-" * 40)

# First message
result1 = buffer_app_no_memory.invoke({
    "messages": [HumanMessage(content="Hi! My name is Alice and I'm a software engineer.")]
})
print(f"User: Hi! My name is Alice and I'm a software engineer.")
print(f"AI: {result1['messages'][-1].content}\n")

# Second message - new invocation, no memory of previous
result2 = buffer_app_no_memory.invoke({
    "messages": [HumanMessage(content="What's my name?")]
})
print(f"User: What's my name?")
print(f"AI: {result2['messages'][-1].content}")
print("\n❌ Notice: The agent doesn't remember Alice because memory isn't persisted!")

Test WITHOUT persistence:
----------------------------------------
User: Hi! My name is Alice and I'm a software engineer.
AI: Hi Alice! Nice to meet you — software engineering is a great field. How can I help you today?

User: What's my name?
AI: I don’t know your name from the information provided. If you’d like, you can tell me and I can use it.

❌ Notice: The agent doesn't remember Alice because memory isn't persisted!


## Part 3: LangGraph Checkpointing for Persistent Memory

LangGraph's checkpointing allows us to persist state between invocations.

In [15]:
# Create a memory checkpointer
memory_saver = MemorySaver()

# Compile WITH checkpointing
buffer_app_with_memory = buffer_graph.compile(checkpointer=memory_saver)

print("Buffer memory graph compiled WITH persistence!")

Buffer memory graph compiled WITH persistence!


In [16]:
# Test WITH persistence
print("Test WITH persistence:")
print("-" * 40)

# Thread ID identifies the conversation
thread_id = "alice-conversation-1"
config = {"configurable": {"thread_id": thread_id}}

# First message
result1 = buffer_app_with_memory.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alice and I'm a software engineer.")]},
    config=config
)
print(f"User: Hi! My name is Alice and I'm a software engineer.")
print(f"AI: {result1['messages'][-1].content}\n")

# Second message - same thread_id, memory persists!
result2 = buffer_app_with_memory.invoke(
    {"messages": [HumanMessage(content="What's my name and what do I do?")]},
    config=config
)
print(f"User: What's my name and what do I do?")
print(f"AI: {result2['messages'][-1].content}")
print("\n✅ The agent remembers Alice!")

Test WITH persistence:
----------------------------------------
User: Hi! My name is Alice and I'm a software engineer.
AI: Hi Alice! Nice to meet you. As a software engineer, you’re in good company here—happy to help with coding, debugging, architecture, tools, or anything else you’re working on.

User: What's my name and what do I do?
AI: Your name is Alice, and you’re a software engineer.

✅ The agent remembers Alice!


In [17]:
# Different thread_id = different conversation
print("\nTest with DIFFERENT thread:")
print("-" * 40)

different_thread = {"configurable": {"thread_id": "bob-conversation-1"}}

result3 = buffer_app_with_memory.invoke(
    {"messages": [HumanMessage(content="Do you remember my name?")]},
    config=different_thread
)
print(f"User: Do you remember my name?")
print(f"AI: {result3['messages'][-1].content}")
print("\n✅ Different thread = separate memory (doesn't know about Alice)")


Test with DIFFERENT thread:
----------------------------------------
User: Do you remember my name?
AI: I don’t actually know your name unless you’ve told me in this chat. If you want, you can tell me and I’ll use it from here on.

✅ Different thread = separate memory (doesn't know about Alice)


In [18]:
# View the conversation history stored in memory
print("\nConversation history for Alice's thread:")
print("-" * 40)

# Get the state for Alice's thread
state_snapshot = buffer_app_with_memory.get_state(config)

for i, msg in enumerate(state_snapshot.values["messages"]):
    role = "User" if isinstance(msg, HumanMessage) else "AI"
    print(f"{i+1}. [{role}]: {msg.content[:100]}..." if len(msg.content) > 100 else f"{i+1}. [{role}]: {msg.content}")


Conversation history for Alice's thread:
----------------------------------------
1. [User]: Hi! My name is Alice and I'm a software engineer.
2. [AI]: Hi Alice! Nice to meet you. As a software engineer, you’re in good company here—happy to help with c...
3. [User]: What's my name and what do I do?
4. [AI]: Your name is Alice, and you’re a software engineer.


## Part 4: Window Memory (Limited Buffer)

To prevent context overflow, we can limit memory to the last N messages.

In [19]:
# State with window limit
class WindowMemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    window_size: int

def window_agent(state: WindowMemoryState) -> dict:
    """Agent that only sees the last N messages."""
    all_messages = state["messages"]
    window_size = state.get("window_size", 6)  # Default to 6 messages (3 exchanges)
    
    # Keep only the last window_size messages
    windowed_messages = all_messages[-window_size:] if len(all_messages) > window_size else all_messages
    
    full_messages = [
        SystemMessage(content=f"You are a helpful assistant. You can see the last {len(windowed_messages)} messages."),
        *windowed_messages
    ]
    
    response = llm.invoke(full_messages)
    return {"messages": [response]}

# Build graph
window_graph = StateGraph(WindowMemoryState)
window_graph.add_node("agent", window_agent)
window_graph.add_edge(START, "agent")
window_graph.add_edge("agent", END)

window_memory = MemorySaver()
window_app = window_graph.compile(checkpointer=window_memory)

print("Window memory graph compiled!")

Window memory graph compiled!


In [20]:
# Test window memory
print("Test Window Memory (window_size=4):")
print("-" * 40)

config = {"configurable": {"thread_id": "window-test"}}

# Conversation with many turns
conversations = [
    "My name is Charlie.",
    "I work at Google.",
    "I live in San Francisco.",
    "I have a dog named Max.",
    "What's my name and where do I work?"
]

for msg in conversations:
    result = window_app.invoke(
        {"messages": [HumanMessage(content=msg)], "window_size": 4},
        config=config
    )
    print(f"User: {msg}")
    print(f"AI: {result['messages'][-1].content}\n")

print("Notice: With window_size=4, older messages may be forgotten!")

Test Window Memory (window_size=4):
----------------------------------------
User: My name is Charlie.
AI: Nice to meet you, Charlie!

User: I work at Google.
AI: Got it — you work at Google, Charlie.

User: I live in San Francisco.
AI: Got it — you live in San Francisco, Charlie.

User: I have a dog named Max.
AI: Got it — you have a dog named Max, Charlie.

User: What's my name and where do I work?
AI: I only know that your name is **Charlie**. I don’t have any information about where you work.

Notice: With window_size=4, older messages may be forgotten!


## Part 5: Summary Memory

Compress older conversations into summaries to save space while retaining key information.

In [21]:
# State with summary
class SummaryMemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    summary: str
    message_count: int
    summarize_threshold: int

def summarize_conversation(messages: list) -> str:
    """Use LLM to summarize the conversation."""
    if not messages:
        return ""
    
    conversation_text = "\n".join([
        f"{'User' if isinstance(m, HumanMessage) else 'Assistant'}: {m.content}"
        for m in messages
    ])
    
    summary_prompt = f"""Summarize this conversation, keeping key facts about the user:

{conversation_text}

Summary (be concise, focus on important facts):"""
    
    response = llm.invoke([HumanMessage(content=summary_prompt)])
    return response.content

def summary_agent(state: SummaryMemoryState) -> dict:
    """Agent that uses summary for older context."""
    messages = state["messages"]
    summary = state.get("summary", "")
    message_count = state.get("message_count", 0) + 1
    threshold = state.get("summarize_threshold", 6)
    
    # Build context with summary if available
    system_content = "You are a helpful assistant."
    if summary:
        system_content += f"\n\nPrevious conversation summary:\n{summary}"
    
    # Use only recent messages + summary
    recent_messages = messages[-4:] if len(messages) > 4 else messages
    
    full_messages = [
        SystemMessage(content=system_content),
        *recent_messages
    ]
    
    response = llm.invoke(full_messages)
    
    # Check if we need to create a new summary
    new_summary = summary
    if message_count >= threshold and message_count % threshold == 0:
        print(f"   📝 Creating summary (message count: {message_count})...")
        # Summarize older messages
        older_messages = messages[:-4] if len(messages) > 4 else []
        if older_messages:
            new_summary = summarize_conversation(older_messages)
            if summary:
                new_summary = f"{summary}\n\nRecent: {new_summary}"
    
    return {
        "messages": [response],
        "summary": new_summary,
        "message_count": message_count
    }

# Build graph
summary_graph = StateGraph(SummaryMemoryState)
summary_graph.add_node("agent", summary_agent)
summary_graph.add_edge(START, "agent")
summary_graph.add_edge("agent", END)

summary_memory = MemorySaver()
summary_app = summary_graph.compile(checkpointer=summary_memory)

print("Summary memory graph compiled!")

Summary memory graph compiled!


In [22]:
# Test summary memory
print("Test Summary Memory:")
print("-" * 40)

config = {"configurable": {"thread_id": "summary-test"}}

conversations = [
    "Hi, I'm Diana.",
    "I'm a data scientist at Meta.",
    "I specialize in recommendation systems.",
    "I have 5 years of experience.",
    "I'm interested in reinforcement learning.",
    "Can you summarize what you know about me?"  # Should trigger summary
]

for i, msg in enumerate(conversations):
    result = summary_app.invoke(
        {
            "messages": [HumanMessage(content=msg)],
            "summary": "",
            "message_count": 0,
            "summarize_threshold": 6
        },
        config=config
    )
    print(f"User: {msg}")
    print(f"AI: {result['messages'][-1].content}\n")

Test Summary Memory:
----------------------------------------
User: Hi, I'm Diana.
AI: Hi Diana — nice to meet you! How can I help today?

User: I'm a data scientist at Meta.
AI: That’s great, Diana — Meta has some fascinating scale and data challenges.

What would you like help with today?

User: I specialize in recommendation systems.
AI: Nice — recommendation systems are a great specialty.

I can help with things like:
- ranking / retrieval / candidate generation
- offline vs online evaluation
- exploration-exploitation
- feature engineering and embeddings
- bias, feedback loops, and cold start
- A/B testing and metric design
- system design for large-scale recsys
- debugging model regressions

What are you working on right now?

User: I have 5 years of experience.
AI: That’s solid experience — enough to own important parts of a recommender stack end to end.

If you’re preparing for interviews or leveling up, I can help you with:
- senior-level recsys interview questions
- system de

In [23]:
# Check the stored summary
state = summary_app.get_state(config)
print("\nStored Summary:")
print("-" * 40)
print(state.values.get("summary", "No summary yet"))


Stored Summary:
----------------------------------------



## Part 6: Entity Memory (User Profile)

Track specific information about entities (like users) for personalization.

In [24]:
# State with entity/profile tracking
class ProfileMemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    user_profile: dict

def extract_user_info(message: str, current_profile: dict) -> dict:
    """Use LLM to extract user information from a message."""
    extraction_prompt = f"""Extract any personal information about the user from this message.
Current known profile: {json.dumps(current_profile)}

Message: {message}

Return a JSON object with any NEW information found. Use these keys when applicable:
- name: user's name
- occupation: job/profession
- company: where they work
- location: where they live
- interests: list of interests
- preferences: any stated preferences

Return only the JSON object, nothing else. If no new info, return {{}}"""
    
    response = llm.invoke([HumanMessage(content=extraction_prompt)])
    
    try:
        # Try to parse the response as JSON
        content = response.content.strip()
        if content.startswith("```"):
            content = content.split("```")[1]
            if content.startswith("json"):
                content = content[4:]
        new_info = json.loads(content)
        return new_info
    except:
        return {}

def profile_agent(state: ProfileMemoryState) -> dict:
    """Agent that maintains a user profile."""
    messages = state["messages"]
    profile = state.get("user_profile", {})
    
    # Extract info from the latest user message
    latest_message = messages[-1]
    if isinstance(latest_message, HumanMessage):
        new_info = extract_user_info(latest_message.content, profile)
        if new_info:
            # Merge new info with existing profile
            for key, value in new_info.items():
                if key == "interests" and key in profile:
                    # Append to interests list
                    existing = profile[key] if isinstance(profile[key], list) else [profile[key]]
                    new_values = value if isinstance(value, list) else [value]
                    profile[key] = list(set(existing + new_values))
                else:
                    profile[key] = value
            print(f"   👤 Profile updated: {list(new_info.keys())}")
    
    # Build context with profile
    system_content = "You are a personalized assistant."
    if profile:
        system_content += f"\n\nUser profile:\n{json.dumps(profile, indent=2)}"
        system_content += "\n\nUse this information to personalize your responses."
    
    full_messages = [
        SystemMessage(content=system_content),
        *messages[-6:]  # Keep last 6 messages
    ]
    
    response = llm.invoke(full_messages)
    
    return {
        "messages": [response],
        "user_profile": profile
    }

# Build graph
profile_graph = StateGraph(ProfileMemoryState)
profile_graph.add_node("agent", profile_agent)
profile_graph.add_edge(START, "agent")
profile_graph.add_edge("agent", END)

profile_memory = MemorySaver()
profile_app = profile_graph.compile(checkpointer=profile_memory)

print("Profile memory graph compiled!")

Profile memory graph compiled!


In [25]:
# Test profile memory
print("Test Profile/Entity Memory:")
print("-" * 40)

config = {"configurable": {"thread_id": "profile-test"}}

conversations = [
    "Hi! I'm Emma and I work as a product manager.",
    "I'm based in Seattle and work at Amazon.",
    "I'm really interested in AI and machine learning.",
    "I also love hiking and photography.",
    "Can you recommend something for me based on what you know about me?"
]

for msg in conversations:
    result = profile_app.invoke(
        {"messages": [HumanMessage(content=msg)], "user_profile": {}},
        config=config
    )
    print(f"User: {msg}")
    print(f"AI: {result['messages'][-1].content}\n")

Test Profile/Entity Memory:
----------------------------------------
   👤 Profile updated: ['name', 'occupation']
User: Hi! I'm Emma and I work as a product manager.
AI: Hi Emma! Nice to meet you — as a product manager, you probably have a lot going on. I’m here to help with writing, brainstorming, prioritization, analysis, meeting prep, or anything else you need.

   👤 Profile updated: ['company', 'location']
User: I'm based in Seattle and work at Amazon.
AI: Got it, Emma — Seattle and Amazon, thanks for sharing. I’ll keep that in mind as we chat. If you want help with anything work-related, especially product thinking or communication, just let me know.

   👤 Profile updated: ['interests']
User: I'm really interested in AI and machine learning.
AI: That’s a great fit, Emma — AI and machine learning are especially relevant for product work right now. I can help with things like:

- explaining ML/AI concepts clearly
- brainstorming AI product ideas
- writing PRDs or strategy docs
- eva

In [26]:
# View the user profile
state = profile_app.get_state(config)
print("\nUser Profile:")
print("-" * 40)
print(json.dumps(state.values.get("user_profile", {}), indent=2))


User Profile:
----------------------------------------
{}


## Part 7: Combining Memory Types

For production systems, combine multiple memory strategies.

In [27]:
# Combined memory state
class CombinedMemoryState(TypedDict):
    messages: Annotated[list, add_messages]
    user_profile: dict  # Long-term entity memory
    conversation_summary: str  # Medium-term summary
    session_facts: list[str]  # Short-term facts from current session

def combined_agent(state: CombinedMemoryState) -> dict:
    """Agent combining multiple memory types."""
    messages = state["messages"]
    profile = state.get("user_profile", {})
    summary = state.get("conversation_summary", "")
    facts = state.get("session_facts", [])
    
    # Build comprehensive system message
    system_parts = ["You are a personalized assistant with excellent memory."]
    
    if profile:
        system_parts.append(f"\nUser Profile (long-term):\n{json.dumps(profile, indent=2)}")
    
    if summary:
        system_parts.append(f"\nConversation Summary (medium-term):\n{summary}")
    
    if facts:
        system_parts.append(f"\nSession Facts (short-term):\n- " + "\n- ".join(facts))
    
    system_parts.append("\nUse all available context to provide personalized, contextual responses.")
    
    full_messages = [
        SystemMessage(content="\n".join(system_parts)),
        *messages[-4:]  # Recent messages
    ]
    
    response = llm.invoke(full_messages)
    
    # Extract any new facts
    latest = messages[-1]
    if isinstance(latest, HumanMessage):
        new_info = extract_user_info(latest.content, profile)
        if new_info:
            for key, value in new_info.items():
                profile[key] = value
            facts.append(f"Learned: {list(new_info.keys())}")
    
    return {
        "messages": [response],
        "user_profile": profile,
        "session_facts": facts[-5:]  # Keep last 5 facts
    }

# Build graph
combined_graph = StateGraph(CombinedMemoryState)
combined_graph.add_node("agent", combined_agent)
combined_graph.add_edge(START, "agent")
combined_graph.add_edge("agent", END)

combined_memory = MemorySaver()
combined_app = combined_graph.compile(checkpointer=combined_memory)

print("Combined memory graph compiled!")

Combined memory graph compiled!


In [28]:
# Test combined memory
print("Test Combined Memory System:")
print("="*50)

config = {"configurable": {"thread_id": "combined-test"}}

conversations = [
    "Hi, I'm Frank, a freelance designer based in Austin.",
    "I specialize in UI/UX design for mobile apps.",
    "I'm currently looking for projects in the healthcare space.",
    "Given what you know about me, what opportunities should I explore?"
]

for msg in conversations:
    result = combined_app.invoke(
        {
            "messages": [HumanMessage(content=msg)],
            "user_profile": {},
            "conversation_summary": "",
            "session_facts": []
        },
        config=config
    )
    print(f"\n👤 User: {msg}")
    print(f"🤖 AI: {result['messages'][-1].content}")

# Show final state
state = combined_app.get_state(config)
print("\n" + "="*50)
print("📊 MEMORY STATE")
print("="*50)
print(f"\nProfile: {json.dumps(state.values.get('user_profile', {}), indent=2)}")
print(f"\nSession Facts: {state.values.get('session_facts', [])}")

Test Combined Memory System:

👤 User: Hi, I'm Frank, a freelance designer based in Austin.
🤖 AI: Nice to meet you, Frank — a freelance designer in Austin. I’ll keep that in mind for future context.

If you’d like, I can also help with:
- portfolio or bio writing
- client proposals and emails
- branding feedback
- pricing/package ideas
- project organization or productivity



👤 User: I specialize in UI/UX design for mobile apps.
🤖 AI: Got it, Frank — you specialize in UI/UX design for mobile apps. That’s a useful focus, especially for portfolio positioning, case studies, and client outreach.

If you want, I can help you turn that into:
- a sharper one-line bio
- a portfolio headline
- a short intro for LinkedIn or your website
- a pitch tailored to app startups or founders

👤 User: I'm currently looking for projects in the healthcare space.
🤖 AI: Understood — you’re a mobile UI/UX designer in Austin, and you’re currently looking for healthcare projects.

That’s a strong niche. I can he

## 🎯 Challenge Exercises

### Challenge 1: Persistent Storage

Modify the profile memory to persist to a JSON file so it survives between sessions.

In [ ]:
# TODO: Implement file-based persistence
# 1. Save user_profile to a JSON file after each update
# 2. Load the profile from file when starting a new session
# 3. Handle the case where the file doesn't exist yet

import json
from pathlib import Path

PROFILE_FILE = "user_profiles.json"

def save_profile(user_id: str, profile: dict):
    """Save profile to file."""
    # Your implementation here
    pass

def load_profile(user_id: str) -> dict:
    """Load profile from file."""
    # Your implementation here
    pass

# Your implementation here

### Challenge 2: Forgetting Mechanism

Implement a way for users to ask the agent to forget specific information.

In [ ]:
# TODO: Implement selective forgetting
# 1. Detect when user asks to forget something ("forget my name", "delete my location")
# 2. Remove the specific field from the profile
# 3. Confirm the deletion

def handle_forget_request(message: str, profile: dict) -> tuple[dict, str]:
    """Handle a request to forget information.
    Returns (updated_profile, confirmation_message)
    """
    # Your implementation here
    pass

### Challenge 3: Semantic Memory with Embeddings

Implement vector-based memory to retrieve relevant past conversations.

In [ ]:
# TODO: Implement semantic memory
# 1. Store conversation snippets with embeddings
# 2. When processing a new message, retrieve the most similar past conversations
# 3. Include relevant past context in the prompt

# Hint: You can use OpenAI embeddings or a local embedding model
# from langchain_openai import OpenAIEmbeddings

# Your implementation here

## Summary

In this exercise, you learned:

1. **Buffer Memory**: Simple but can overflow
2. **Checkpointing**: Persist state with thread IDs
3. **Window Memory**: Bounded recent context
4. **Summary Memory**: Compress older conversations
5. **Entity Memory**: Track user profiles
6. **Combined Memory**: Use multiple strategies together

These patterns enable building assistants that truly remember and personalize!